# EWS Multivariate Anomaly Detection — Simulation Notebook

Bu notebook tüm pipeline'ı adım adım simüle eder:

1. **Veri Üretimi** — Sentetik normal + anomalili veri
2. **EDA** — Feature dağılımları, korelasyonlar
3. **Model Eğitimi** — Autoencoder + Isolation Forest + Mahalanobis
4. **Skorlama** — Ensemble skor + human-readable neden
5. **Değerlendirme** — Precision, recall, bant analizi
6. **Tek Müşteri Deep Dive** — Bir müşteriyi detaylı incele

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

from config import ALL_FEATURES, FEATURE_LABELS, PAYMENT_FEATURES, UTILIZATION_FEATURES, TRANSACTION_FEATURES, BALANCE_FEATURES, TREND_FEATURES
from generate_data import generate_normal_data, generate_inference_data
from model import ExplainableEnsemble

print(f'Feature sayisi: {len(ALL_FEATURES)}')
print(f'Kategoriler: Payment({len(PAYMENT_FEATURES)}), Utilization({len(UTILIZATION_FEATURES)}), Transaction({len(TRANSACTION_FEATURES)}), Balance({len(BALANCE_FEATURES)}), Trend({len(TREND_FEATURES)})')

---
## 1. Veri Üretimi

In [ ]:
# Training data: 5000 normal musteri
train_df = generate_normal_data(5000)
print(f'Training: {train_df.shape}')
train_df.head()

In [ ]:
# Inference data: 5000 musteri, 200 anomali enjekte edilmis
inf_df, labels = generate_inference_data(5000)
print(f'Inference: {inf_df.shape}')
print(f'\nAnomali dagilimi:')
print(labels['anomaly_type'].value_counts())

---
## 2. EDA — Feature Dağılımları ve Korelasyonlar

In [ ]:
# Training data istatistikleri
train_df[ALL_FEATURES].describe().round(2)

In [ ]:
# Korelasyon matrisi — onemli feature iliskileri
corr = train_df[ALL_FEATURES].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            xticklabels=[f[:15] for f in ALL_FEATURES],
            yticklabels=[f[:15] for f in ALL_FEATURES])
ax.set_title('Feature Korelasyon Matrisi (Training Data)', fontsize=14)
plt.tight_layout()
plt.show()

# En guclu korelasyonlar
strong = []
for i in range(len(ALL_FEATURES)):
    for j in range(i+1, len(ALL_FEATURES)):
        if abs(corr.iloc[i, j]) > 0.3:
            strong.append((ALL_FEATURES[i], ALL_FEATURES[j], round(corr.iloc[i, j], 3)))
strong.sort(key=lambda x: abs(x[2]), reverse=True)
print('Guclu korelasyonlar (|r| > 0.3):')
for f1, f2, r in strong[:10]:
    print(f'  {f1:35s} <-> {f2:35s}  r={r:+.3f}')

In [ ]:
# Normal vs Anomali karsilastirmasi — secili feature'lar
inf_merged = inf_df.merge(labels, on='customer_id')

key_features = ['utilization_ratio', 'days_past_due', 'min_payment_only_count_6m',
                'txn_count_monthly', 'outstanding_balance', 'cash_advance_to_spending_ratio']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feat in zip(axes.flat, key_features):
    for atype, color in [('NORMAL', '#2ecc71'), ('A_UNIVARIATE', '#e74c3c'),
                          ('B_MULTIVARIATE', '#e67e22'), ('C_SUBTLE_DRIFT', '#9b59b6')]:
        subset = inf_merged[inf_merged['anomaly_type'] == atype][feat]
        if len(subset) > 0:
            ax.hist(subset, bins=40, alpha=0.5, label=atype, density=True, color=color)
    ax.set_title(FEATURE_LABELS.get(feat, feat), fontsize=10)
    ax.legend(fontsize=7)
fig.suptitle('Normal vs Anomali Dagilim Karsilastirmasi', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Multivariate anomali ornegi: util yuksek ama txn dusuk
fig, ax = plt.subplots(figsize=(10, 7))
for atype, color, marker in [('NORMAL', '#2ecc71', '.'), ('B_MULTIVARIATE', '#e74c3c', 'X')]:
    subset = inf_merged[inf_merged['anomaly_type'] == atype]
    ax.scatter(subset['txn_count_monthly'], subset['utilization_ratio'],
               c=color, marker=marker, alpha=0.4, s=30 if atype=='NORMAL' else 80, label=atype)
ax.set_xlabel('Aylik Islem Sayisi')
ax.set_ylabel('Limit Kullanim Orani')
ax.set_title('Multivariate Anomali: Yuksek Kullanim + Dusuk Islem Sayisi', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Model Eğitimi

3 model parallel eğitilir:
- **Autoencoder** (35 → 24 → 12 → 6 → 12 → 24 → 35)
- **Isolation Forest** (200 tree)
- **Mahalanobis** (Robust covariance)

In [ ]:
model = ExplainableEnsemble()
model.fit(train_df)

In [ ]:
# AE'nin ogrendigi: training datadaki reconstruction error dagilimi
from sklearn.preprocessing import StandardScaler
import torch

X_train_scaled = model.scaler.transform(train_df[ALL_FEATURES].fillna(0))
train_errors = model._ae_total_error(X_train_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_errors, bins=80, color='#3498db', alpha=0.7, edgecolor='white')
axes[0].axvline(np.percentile(train_errors, 95), color='orange', linestyle='--', label='p95')
axes[0].axvline(np.percentile(train_errors, 99), color='red', linestyle='--', label='p99')
axes[0].set_title('AE Reconstruction Error Dagilimi (Training)', fontsize=12)
axes[0].set_xlabel('Total Reconstruction Error')
axes[0].legend()

# Mahalanobis distance dagilimi
train_md = model._mahal_distances(X_train_scaled)
axes[1].hist(train_md, bins=80, color='#9b59b6', alpha=0.7, edgecolor='white')
axes[1].axvline(np.percentile(train_md, 95), color='orange', linestyle='--', label='p95')
axes[1].axvline(np.percentile(train_md, 99), color='red', linestyle='--', label='p99')
axes[1].set_title('Mahalanobis Distance Dagilimi (Training)', fontsize=12)
axes[1].set_xlabel('Mahalanobis Distance')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'AE Error  — p50: {np.median(train_errors):.2f}, p95: {np.percentile(train_errors, 95):.2f}, p99: {np.percentile(train_errors, 99):.2f}')
print(f'Mahal Dist — p50: {np.median(train_md):.2f}, p95: {np.percentile(train_md, 95):.2f}, p99: {np.percentile(train_md, 99):.2f}')

---
## 4. Skorlama (Inference)

In [ ]:
results = model.predict(inf_df, top_n=3)
print(f'Skorlanan musteri: {len(results)}')
print(f'\nBant dagilimi:')
print(results['alert_band'].value_counts().to_frame('count'))

In [ ]:
# Skor dagilimi — genel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
colors = {'NORMAL': '#2ecc71', 'SARI': '#f1c40f', 'TURUNCU': '#e67e22', 'KIRMIZI': '#e74c3c'}
for band in ['NORMAL', 'SARI', 'TURUNCU', 'KIRMIZI']:
    subset = results[results['alert_band'] == band]['anomaly_score']
    axes[0].hist(subset, bins=40, alpha=0.6, color=colors[band], label=f'{band} ({len(subset)})', edgecolor='white')
axes[0].set_title('Anomaly Score Dagilimi (Bant Bazli)', fontsize=12)
axes[0].set_xlabel('Anomaly Score (0-100)')
axes[0].legend()

# Alt modellerin karsilastirmasi
axes[1].scatter(results['ae_score'], results['if_score'], c=results['anomaly_score'],
                cmap='RdYlGn_r', alpha=0.4, s=15)
axes[1].set_xlabel('AE Score')
axes[1].set_ylabel('IF Score')
axes[1].set_title('AE vs IF Skor Karsilastirmasi', fontsize=12)
plt.colorbar(axes[1].collections[0], ax=axes[1], label='Ensemble Score')

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 alert — human readable
top10 = results.head(10)

for _, row in top10.iterrows():
    band_icon = {'KIRMIZI': '🔴', 'TURUNCU': '🟠', 'SARI': '🟡'}.get(row['alert_band'], '⚪')
    print(f"\n{band_icon} {row['customer_id']}  |  Skor: {row['anomaly_score']}  |  {row['alert_band']}")
    print(f"   AE: {row['ae_score']}  |  IF: {row['if_score']}  |  MD: {row['md_score']}")
    for feat, d in row['detay'].items():
        icon = '↑' if d['degisim_pct'] > 0 else '↓'
        print(f"   • {d['label']}: {d['beklenen']} → {d['gerceklesen']} ({icon}%{abs(d['degisim_pct']):.0f}, katki %{d['katki_pct']:.0f})")

---
## 5. Değerlendirme — Precision, Recall, Bant Analizi

In [ ]:
# Labels ile birlestir
eval_df = results.merge(labels, on='customer_id')

# Bant bazli precision
print('='*65)
print('BANT BAZLI PERFORMANS')
print('='*65)
for band in ['KIRMIZI', 'TURUNCU', 'SARI']:
    in_band = eval_df[eval_df['alert_band'] == band]
    n = len(in_band)
    true = in_band['is_anomaly'].sum()
    prec = true / n * 100 if n > 0 else 0
    by_type = in_band[in_band['is_anomaly']].groupby('anomaly_type').size()
    print(f'\n  {band}:')
    print(f'    Alert: {n}, Gercek anomali: {true}, Precision: %{prec:.1f}')
    for t, c in by_type.items():
        print(f'      {t}: {c}')

In [ ]:
# Recall — anomali tipine gore
print('\n' + '='*65)
print('ANOMALI TIPI BAZLI RECALL')
print('='*65)

alerted = eval_df[eval_df['alert_band'].isin(['KIRMIZI', 'TURUNCU', 'SARI'])]

recall_data = []
for atype in ['A_UNIVARIATE', 'B_MULTIVARIATE', 'C_SUBTLE_DRIFT']:
    total = len(eval_df[eval_df['anomaly_type'] == atype])
    caught = len(alerted[alerted['anomaly_type'] == atype])
    recall = caught / total * 100 if total > 0 else 0
    recall_data.append({'Tip': atype, 'Toplam': total, 'Yakalanan': caught, 'Recall %': round(recall, 1)})
    print(f'  {atype}: {caught}/{total} (%{recall:.1f})')

total_a = len(eval_df[eval_df['is_anomaly']])
total_c = len(alerted[alerted['is_anomaly']])
print(f'\n  TOPLAM: {total_c}/{total_a} (%{total_c/total_a*100:.1f})')

In [ ]:
# Precision-Recall esik analizi
thresholds = list(range(40, 100, 5))
pr_data = []

total_anomalies = eval_df['is_anomaly'].sum()

for t in thresholds:
    flagged = eval_df[eval_df['anomaly_score'] >= t]
    n_flagged = len(flagged)
    n_true = flagged['is_anomaly'].sum()
    precision = n_true / n_flagged * 100 if n_flagged > 0 else 0
    recall = n_true / total_anomalies * 100 if total_anomalies > 0 else 0
    pr_data.append({'Esik': t, 'Alert Sayisi': n_flagged, 'Dogru Alert': n_true,
                    'Precision %': round(precision, 1), 'Recall %': round(recall, 1)})

pr_df = pd.DataFrame(pr_data)
print('\nEsik Analizi:')
print(pr_df.to_string(index=False))

# Gorsellestrme
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(pr_df['Esik'], pr_df['Precision %'], 'o-', color='#e74c3c', linewidth=2, label='Precision %')
ax1.plot(pr_df['Esik'], pr_df['Recall %'], 's-', color='#3498db', linewidth=2, label='Recall %')
ax1.set_xlabel('Esik Degeri', fontsize=12)
ax1.set_ylabel('%', fontsize=12)
ax1.set_title('Precision vs Recall — Farkli Esik Degerlerinde', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Bant sinirlari
for val, label, color in [(60, 'SARI', '#f1c40f'), (75, 'TURUNCU', '#e67e22'), (90, 'KIRMIZI', '#e74c3c')]:
    ax1.axvline(val, color=color, linestyle='--', alpha=0.5)
    ax1.text(val+0.5, 5, label, color=color, fontsize=9)

ax2 = ax1.twinx()
ax2.bar(pr_df['Esik'], pr_df['Alert Sayisi'], alpha=0.15, color='gray', width=4, label='Alert Sayisi')
ax2.set_ylabel('Alert Sayisi', fontsize=12)
ax2.legend(loc='upper center', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Anomali tipine gore skor dagilimi
fig, ax = plt.subplots(figsize=(12, 5))

type_colors = {'NORMAL': '#2ecc71', 'A_UNIVARIATE': '#e74c3c',
               'B_MULTIVARIATE': '#e67e22', 'C_SUBTLE_DRIFT': '#9b59b6'}

for atype, color in type_colors.items():
    subset = eval_df[eval_df['anomaly_type'] == atype]['anomaly_score']
    ax.hist(subset, bins=40, alpha=0.5, color=color, label=f'{atype} (n={len(subset)})', density=True)

ax.axvline(60, color='#f1c40f', linestyle='--', alpha=0.7, label='SARI siniri')
ax.axvline(75, color='#e67e22', linestyle='--', alpha=0.7, label='TURUNCU siniri')
ax.axvline(90, color='#e74c3c', linestyle='--', alpha=0.7, label='KIRMIZI siniri')
ax.set_xlabel('Anomaly Score')
ax.set_title('Anomali Tipine Gore Skor Dagilimi', fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 6. Tek Müşteri Deep Dive

Bir anomali müşterisini detaylı inceleyelim: 3 modelin her birinin ne dediğini, katkı paylarını, beklenen vs gerçekleşen değerleri görelim.

In [ ]:
# KIRMIZI banttan bir musteri sec
red_alerts = eval_df[(eval_df['alert_band'] == 'KIRMIZI') & (eval_df['is_anomaly'])]
sample = red_alerts.iloc[0]
cust_id = sample['customer_id']
cust_type = sample['anomaly_type']

print(f'Musteri: {cust_id}')
print(f'Anomali Tipi: {cust_type}')
print(f'Ensemble Skor: {sample["anomaly_score"]}')
print(f'Bant: {sample["alert_band"]}')
print(f'\nAlt Model Skorlari:')
print(f'  AE:  {sample["ae_score"]}')
print(f'  IF:  {sample["if_score"]}')
print(f'  MD:  {sample["md_score"]}')
print(f'\nNeden:')
for feat, d in sample['detay'].items():
    icon = '↑' if d['degisim_pct'] > 0 else '↓'
    print(f'  • {d["label"]}: {d["beklenen"]} → {d["gerceklesen"]} ({icon}%{abs(d["degisim_pct"]):.0f}, katki %{d["katki_pct"]:.0f})')

In [ ]:
# Bu musterinin tum feature'lari icin: 3 modelin katki paylari
cust_row = inf_df[inf_df['customer_id'] == cust_id]
X_cust = model.scaler.transform(cust_row[ALL_FEATURES].fillna(0))

ae_c = model._ae_contribution(X_cust)[0]
if_c = model._if_contribution(X_cust)[0]
md_c = model._md_contribution(X_cust)[0]
unified = 0.5 * ae_c + 0.3 * if_c + 0.2 * md_c

contrib_df = pd.DataFrame({
    'Feature': ALL_FEATURES,
    'Label': [FEATURE_LABELS.get(f, f) for f in ALL_FEATURES],
    'AE %': (ae_c * 100).round(1),
    'IF %': (if_c * 100).round(1),
    'MD %': (md_c * 100).round(1),
    'Birlesik %': (unified * 100).round(1),
}).sort_values('Birlesik %', ascending=False)

print(f'\n{cust_id} — 3 Model Katki Paylari (top 10):')
print(contrib_df.head(10).to_string(index=False))

In [ ]:
# Gorsel: Top 10 feature'in 3 modeldeki katki paylari
top10_contrib = contrib_df.head(10)

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(top10_contrib))
w = 0.25

ax.bar(x - w, top10_contrib['AE %'], w, label='Autoencoder', color='#3498db', alpha=0.8)
ax.bar(x, top10_contrib['IF %'], w, label='Isolation Forest', color='#e67e22', alpha=0.8)
ax.bar(x + w, top10_contrib['MD %'], w, label='Mahalanobis', color='#9b59b6', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels([l[:20] for l in top10_contrib['Label']], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Katki %')
ax.set_title(f'{cust_id} — Feature Katki Paylari (3 Model)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Beklenen vs Gerceklesen — radar chart
expected_X = model._ae_reconstruct(X_cust)

top5_idx = contrib_df.head(5)['Feature'].values
actual_vals = []
expected_vals = []
feat_labels = []

for feat in top5_idx:
    j = ALL_FEATURES.index(feat)
    act = X_cust[0, j] * model.scaler.scale_[j] + model.scaler.mean_[j]
    exp = expected_X[0, j] * model.scaler.scale_[j] + model.scaler.mean_[j]
    actual_vals.append(act)
    expected_vals.append(exp)
    feat_labels.append(FEATURE_LABELS.get(feat, feat)[:18])

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(feat_labels))
ax.bar(x - 0.2, expected_vals, 0.35, label='Beklenen (AE)', color='#3498db', alpha=0.7)
ax.bar(x + 0.2, actual_vals, 0.35, label='Gerceklesen', color='#e74c3c', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(feat_labels, rotation=25, ha='right', fontsize=9)
ax.set_title(f'{cust_id} — Beklenen vs Gerceklesen (Top 5 Feature)', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('\nDetay:')
for feat, exp, act in zip(top5_idx, expected_vals, actual_vals):
    label = FEATURE_LABELS.get(feat, feat)
    pct = ((act - exp) / abs(exp) * 100) if abs(exp) > 1e-6 else 0
    icon = '↑' if pct > 0 else '↓'
    print(f'  {label:40s}  beklenen: {exp:>10.2f}  gerceklesen: {act:>10.2f}  ({icon}%{abs(pct):.0f})')

---
## 7. İş Birimi Rapor Formatı

Haftalık alert çıktısının iş birimine gönderilecek hali.

In [ ]:
# Haftalik alert raporu
alert_df = results[results['alert_band'].isin(['KIRMIZI', 'TURUNCU', 'SARI'])].copy()

# Detayi duz sutunlara ac
report_rows = []
for _, row in alert_df.iterrows():
    base = {
        'Musteri': row['customer_id'],
        'Skor': row['anomaly_score'],
        'Bant': row['alert_band'],
    }
    reasons = []
    for i, (feat, d) in enumerate(row['detay'].items(), 1):
        icon = '↑' if d['degisim_pct'] > 0 else '↓'
        reasons.append(f"{d['label']}: {d['beklenen']}→{d['gerceklesen']} ({icon}%{abs(d['degisim_pct']):.0f})")
        base[f'Neden_{i}'] = reasons[-1]
        base[f'Katki_{i}'] = f"%{d['katki_pct']}"
    report_rows.append(base)

report = pd.DataFrame(report_rows)
print(f'Toplam alert: {len(report)}')
print(f'  KIRMIZI: {len(report[report["Bant"]=="KIRMIZI"])}')
print(f'  TURUNCU: {len(report[report["Bant"]=="TURUNCU"])}')
print(f'  SARI:    {len(report[report["Bant"]=="SARI"])}')
print()
report.head(10)

In [ ]:
# Excel'e kaydet (is birimine gonderilecek format)
report.to_excel('output/haftalik_alert_raporu.xlsx', index=False, sheet_name='Alerts')
print('Kaydedildi: output/haftalik_alert_raporu.xlsx')

---
## 8. Özet

| Bileşen | Detay |
|---------|-------|
| Feature | 35 değişken, 5 kategori |
| Model | AE (w=0.5) + IF (w=0.3) + Mahalanobis (w=0.2) |
| Skor | 0-100, ensemble percentile rank |
| Bantlar | NORMAL (<60), SARI (60-75), TURUNCU (75-90), KIRMIZI (90+) |
| Neden | Per-feature contribution → beklenen vs gerçekleşen → Türkçe label |
| Çıktı | Müşteri ID, skor, bant, top-3 neden (human-readable) |